In [1]:
# Setup - disable progress bars before any imports
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TQDM_DISABLE'] = '1'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

# Disable tqdm entirely by patching before imports
import sys

class DummyTqdm:
    def __init__(self, iterable=None, *args, **kwargs):
        self.iterable = iterable
    def __iter__(self):
        return iter(self.iterable) if self.iterable else iter([])
    def __enter__(self):
        return self
    def __exit__(self, *args):
        pass
    def update(self, *args, **kwargs):
        pass
    def close(self, *args, **kwargs):
        pass
    def set_description(self, *args, **kwargs):
        pass

# Create a mock tqdm module
class MockTqdmModule:
    tqdm = DummyTqdm
    def __getattr__(self, name):
        return DummyTqdm

mock_tqdm = MockTqdmModule()
sys.modules['tqdm'] = mock_tqdm
sys.modules['tqdm.auto'] = mock_tqdm
sys.modules['tqdm.std'] = mock_tqdm
sys.modules['tqdm.notebook'] = mock_tqdm

print("Environment prepared with tqdm disabled")

Environment prepared with tqdm disabled


In [2]:
# Set up working directory
os.chdir('/home/smallyan/eval_agent')
sys.path.insert(0, '/net/scratch2/smallyan/filter_eval')
os.chdir('/net/scratch2/smallyan/filter_eval')
print(f"Working directory: {os.getcwd()}")

Working directory: /net/scratch2/smallyan/filter_eval


In [3]:
# Now try loading torch and transformers
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA device: NVIDIA H100 NVL


In [4]:
# Load model with transformers
import transformers
print(f"Transformers version: {transformers.__version__}")

from transformers import AutoModelForCausalLM, AutoTokenizer

model_key = "meta-llama/Llama-3.3-70B-Instruct"
print(f"Loading model: {model_key}...")

model = AutoModelForCausalLM.from_pretrained(
    model_key,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
    local_files_only=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_key, local_files_only=True)
print("Model loaded successfully!")

Error importing huggingface_hub.hf_api: 'type' object is not iterable


TypeError: 'type' object is not iterable